In [0]:
CREATE OR REFRESH LIVE TABLE gold_customer_ltv
COMMENT "Total quantity and order count per customer, current name/email"
TBLPROPERTIES ("quality" = "gold")
AS
SELECT
  c.id                                   AS customer_id,
  c.first_name,
  c.last_name,
  c.email,
  COUNT(o.id)                            AS total_orders,
  SUM(o.quantity)                        AS total_units_purchased
FROM LIVE.silver_orders o
JOIN LIVE.silver_customers c
  ON o.purchaser = c.id
WHERE c.__END_AT IS NULL           -- only the current version of each customer
GROUP BY c.id, c.first_name, c.last_name, c.email;

In [0]:
CREATE OR REFRESH LIVE TABLE gold_order_history_point_in_time
COMMENT "Orders joined to the customer record valid at order time (SCD2 demo)"
TBLPROPERTIES ("quality" = "gold")
AS
SELECT
  o.id                AS order_id,
  o.order_date,
  o.quantity,
  c.id                AS customer_id,
  c.first_name        AS customer_first_name_at_order_time,
  c.last_name         AS customer_last_name_at_order_time,
  c.email             AS customer_email_at_order_time,
  p.name              AS product_name
FROM LIVE.silver_orders o
JOIN LIVE.silver_customers c
  ON o.purchaser = c.id
  AND o.order_date >= c.__START_AT
  AND (c.__END_AT IS NULL OR o.order_date < c.__END_AT)
JOIN LIVE.silver_products p
  ON o.product_id = p.id;
 

In [0]:
CREATE OR REFRESH LIVE TABLE gold_product_performance
COMMENT "Units sold per product, ranked"
TBLPROPERTIES ("quality" = "gold")
AS
SELECT
  p.id                            AS product_id,
  p.name,
  p.weight,
  COUNT(o.id)                     AS order_count,
  COALESCE(SUM(o.quantity), 0)    AS total_units_sold,
  RANK() OVER (ORDER BY COALESCE(SUM(o.quantity), 0) DESC) AS sales_rank
FROM LIVE.silver_products p
LEFT JOIN LIVE.silver_orders o
  ON p.id = o.product_id
GROUP BY p.id, p.name, p.weight;
 

In [0]:
CREATE OR REFRESH LIVE TABLE gold_inventory_status
COMMENT "Current inventory level per product with a low-stock flag"
TBLPROPERTIES ("quality" = "gold")
AS
SELECT
  p.id                AS product_id,
  p.name,
  poh.quantity        AS quantity_on_hand,
  CASE WHEN poh.quantity < 5 THEN TRUE ELSE FALSE END AS is_low_stock
FROM LIVE.silver_products p
JOIN LIVE.silver_products_on_hand poh
  ON p.id = poh.product_id;

In [0]:
CREATE OR REFRESH LIVE TABLE gold_daily_order_volume
COMMENT "Order count and units sold per day"
TBLPROPERTIES ("quality" = "gold")
AS
SELECT
  order_date,
  COUNT(id)        AS order_count,
  SUM(quantity)    AS units_sold
FROM LIVE.silver_orders
GROUP BY order_date
ORDER BY order_date;